# Reviewing the classified data

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

try:
    start = Path(__file__).resolve()
except NameError:
    start = Path.cwd()


supporting_files = next(p / "00-supporting-files" for p in start.parents if (p / "00-supporting-files").exists())
os.listdir(supporting_files)

['data', 'images']

In [3]:
import os
import json

reviews_path = supporting_files / "data" / "reviews"
file_path = os.path.join(reviews_path, 'classified_reviews.json')

with open(file_path, 'r') as json_file:
    classified_data = json.load(json_file)

classified_data[0]

{'uuid': 'b8f0ba8d-2fa5-4261-b48e-b6be38b4fcdb',
 'title': 'A very nice backpack',
 'rating': 4,
 'body_html': "<p>Overall, the best backpack I've owned and will continue to use it well into the future. </p>\n\n<p>Pros:\n<br />Feels very sturdy and durable </p>\n\n<p>Lots of storage space </p>\n\n<p>Straps are quite comfortable </p>\n\n<p>Design is practical and well thought-out </p>\n\n<p>Water-resistance has been excellent so far</p>\n\n<p>Cons:\n<br />The foam shoulder padding on the straps seems to be wearing out very unevenly. I always have the backpack over both shoulders, not sure why the left strap would flatten out while the right still looks new.\n<br />A bit heavy, even while empty</p>\n\n<p>Description advertised it stands up on its own but, I haven't found that to be reliable </p>\n\n<p>I wish it had some kind of reflective spots on the back for cyclists/scooter riders at night</p>\n\n<p>For my use case, a slightly bigger main-area, and a smaller sleeve/laptop area would b

In [4]:
import pandas as pd
df = pd.DataFrame(classified_data)
df.head()

,uuid,title,rating,body_html,body,classification,classification_explanation,sentiment,sentiment_explanation,verified_buyer,...,media_platform_hosted_video_infos,country_code_show_flag,pinned,in_a_group,shop_reply_name,verified_purchase,reviewer_public_slug,reviewer_public_id,collected_source,has_coupon
0,b8f0ba8d-2fa5-4261-b48e-b6be38b4fcdb,A very nice backpack,4,"<p>Overall, the best backpack I've owned and w...","Overall, the best backpack I've owned and will...",Product Review,The text explicitly discusses the pros and con...,3,"While the customer highlights some issues, the...",True,...,[],CA,None,False,lttstore.com,True,NaN,yKYbaJp5,organic,False
1,7d5a2fa4-c6f5-47ac-826e-95971249d735,Title,5,<p>Great backpack.</p>,Great backpack.,General Satisfaction,The review simply states 'Great backpack' indi...,1,The word 'great' clearly expresses a positive ...,True,...,[],CZ,None,False,lttstore.com,True,NaN,Djok4eAk,invite,False
2,93f3034f-ef5b-5720-abe4-ff89a80db2fb,,5,<p>I use it every day and it’s great</p>,I use it every day and it’s great,General Satisfaction,The reviewer states they use the product daily...,1,The phrase 'it’s great' expresses a strongly p...,True,...,[],None,None,False,lttstore.com,True,NaN,None,None,False
3,26e92a79-5272-4e72-935a-a1faa3b2e4c1,Stiff,4,"<p>This badboy is pretty stiff, i could have s...","This badboy is pretty stiff, i could have seen...",Product Functionality & Design,The review focuses on the backpack's compartme...,3,"While the reviewer praises the quality, comfor...",True,...,[],DE,None,False,lttstore.com,True,NaN,WP88RY6R,invite,False
4,dae6c877-9f50-48e2-818f-46e70ed50747,What a backpack!,5,<p>This is an absolutely incredible backpack T...,This is an absolutely incredible backpack The ...,Product Features & Size,"The review focuses on the backpack's size, spa...",100,The language used is overwhelmingly positive (...,True,...,[],US,None,False,lttstore.com,True,NaN,DL7a7jjO,invite,False


In [22]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    counts_df = (
    df.value_counts(["rating","classification", ])
      .rename("count")
      .reset_index()
      .sort_values(["rating","classification"], ascending=[True, True], kind="stable")
      .reset_index(drop=True)
  )

    display(counts_df)



,rating,classification,count
0,1,Complaint,1
1,1,Customer Service,2
2,1,Customer Service/Order Fulfillment,1
3,1,Customer Support and Billing Issues,1
4,1,Delivery Status,2
5,1,Delivery/Order Issue,1
6,1,Manufacturing Defect,1
7,1,Negative Product Experience,1
8,1,Order Delay and Communication Issues,1
9,1,Order Status,1


In [4]:
df["classification"].unique().tolist()

['Product Review',
 'General Satisfaction',
 'Product Functionality & Design',
 'Product Features & Size',
 'Product Features & Functionality',
 'Size and Functionality',
 'Durability & Functionality',
 'Durability and Longevity',
 'Feature Request',
 'Fit & Comfort',
 'Quality and Value',
 'Gift & Price',
 'Overall Quality',
 'Space and Pockets',
 'Price and Value',
 'Product Quality & Durability',
 'Functionality',
 'Durability & Value',
 'Durability & Build Quality',
 'Quality',
 'Durability & Longevity',
 'Capacity and Functionality',
 'Product Quality & Design',
 'Product Description',
 'Warranty & Customer Service',
 'Durability & Water Resistance',
 'Customer Service',
 'Size',
 'Price',
 'Size and Capacity',
 'Quality & Value',
 'Product Quality and Price',
 'Functionality & Capacity',
 'Storage Capacity & Quality',
 'Product Features & Comfort',
 'Product Feedback',
 'Storage and Organization',
 'Positive Review',
 'Size & Capacity',
 'Product Functionality',
 'Functionality &

In [35]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):

    display(nz_long)



,rating,classification,sentiment_bin,count
261,1,Complaint,neg,1
271,1,Customer Service,neu,2
289,1,Customer Service/Order Fulfillment,neu,1
292,1,Customer Support and Billing Issues,neu,1
313,1,Delivery Status,neu,2
328,1,Delivery/Order Issue,neu,1
805,1,Manufacturing Defect,neu,1
832,1,Negative Product Experience,neu,1
844,1,Order Delay and Communication Issues,neu,1
847,1,Order Status,neu,1


In [5]:
import re
import numpy as np
import pandas as pd

# --- 1) Your bucketing rules (ordered = priority for "primary_category") ---
rules = {
    "Shipping/Delivery/Fulfillment": [
        "shipping","delivery","fulfillment","order status","order delay","returns","refund"
    ],
    "Customer Service/Warranty/Billing": [
        "customer service","warranty","billing","website issues","support"
    ],
    "Price/Value/Cost": [
        "price","value","pricing","tax","cost"
    ],
    "Defect/Quality Issues": [
        "defect","issue","smell issue","smell","odor","scent","manufacturing defect",
        "quality control","quality issues","dye bleeding","size misrepresentation"
    ],
    "Quality/Durability/Build": [
        "quality","durability","build quality","sturdiness","reliability","longevity","structure","appearance"
    ],
    "Design/Features/Functionality": [
        "design flaw","design flaws","design suggestion","design feedback","design",
        "feature","functionality","use case","versatility","practicality","instructions",
        "transparency","usability","convenience"
    ],
    "Size/Capacity/Organization/Space": [
        "size","capacity","organization","compartment","space","pockets","storage",
        "compartmentalization","packing"
    ],
    "Comfort/Fit/Ergonomics/Support": [
        "comfort","fit","ergonomics","support","breathability","ventilation","back support","straps"
    ],
    "Travel/Airline/Carry-on": [
        "travel","airline","carry-on","suitcase","frequent traveler","luggage"
    ],
    "Security/Safety": [
        "security","safety","protection","crash protection","pass-through strap"
    ],
    "Water Resistance/Waterproof": [
        "water resistance","waterproof"
    ],
    "Weight/Portability": [
        "weight","portability"
    ],
    "Laptop/Tech": [
        "laptop","tech"
    ],
    "Aesthetics/Style": [
        "aesthetics","style"
    ],
    "Brand/Trust": [
        "brand trust","brand loyalty","brand support"
    ],
    "Packaging/Contents/Missing": [
        "packaging","contents","missing item","missing items","missing accessories"
    ],
    "Comparison/Recommendation": [
        "comparison","recommendation","recommend","compare"," vs ","vs."
    ],
    "Sentiment/Review Meta": [
        "review","positive","negative","mixed","general","overall","neutral","impressions",
        "satisfaction","complaint","interest","purchase intent","surprise","unknown"
    ],
    "Specialized Use/Accessories": [
        "airtag","drone","pilot bag","motorcycle","tracker pocket","pass-through","suitability"
    ],
}
cat_order = list(rules.keys())

# --- 2) Precompile case-insensitive patterns ---
patterns = {
    cat: re.compile("|".join(map(re.escape, kws)), flags=re.I)
    for cat, kws in rules.items()
}

# Assume your column is df["label"]; normalize to string
s = df["classification"].astype(str).str.strip()

# --- 3) Build a boolean match matrix (rows = rows of df, cols = categories) ---
mask_df = pd.DataFrame({cat: s.str.contains(pat, na=False) for cat, pat in patterns.items()})

# --- 4) Primary category by priority (first matching rule) ---
DEFAULT = "Other/Uncategorized"
primary = pd.Series(DEFAULT, index=df.index)
for cat in cat_order:
    primary = np.where((primary == DEFAULT) & mask_df[cat], cat, primary)
primary = pd.Series(primary, index=df.index, name="primary_category")

# --- 5) Secondary categories = all matches except the chosen primary ---
secondary = (
    mask_df.apply(
        lambda row: "; ".join([cat for cat in cat_order if row[cat] and cat != primary.loc[row.name]]),
        axis=1
    )
    .replace("", None)
    .rename("secondary_categories")
)

# --- 6) Attach to your DataFrame ---
df = df.assign(primary_category=primary, secondary_categories=secondary)

# (Optional) quick summary
summary = (
    df["primary_category"].value_counts()
      .rename_axis("primary_category")
      .reset_index(name="count")
)


In [6]:
summary

,primary_category,count
0,Quality/Durability/Build,735
1,Sentiment/Review Meta,665
2,Design/Features/Functionality,654
3,Size/Capacity/Organization/Space,237
4,Other/Uncategorized,234
5,Shipping/Delivery/Fulfillment,145
6,Price/Value/Cost,137
7,Customer Service/Warranty/Billing,55
8,Comfort/Fit/Ergonomics/Support,39
9,Travel/Airline/Carry-on,39


In [ ]:
df.value_counts("sentiment")

sentiment
 4      666
 1      610
 100    511
 3      288
-1      258
 5      242
 0      178
 2      129
 10      49
 75      29
 90      15
-2        5
-10       3
 9        3
 7        3
 8        2
-5        1
 80       1
 95       1
Name: count, dtype: int64

In [7]:
import numpy as np
import pandas as pd

# Parse to numeric
s = pd.to_numeric(df["sentiment"], errors="coerce")

# Heuristic source-scale masks
m_pct   = s >= 11                                   # 0–100-ish "percent-like" (75, 90, 100, ...)
m_0to10 = s.between(6, 10, inclusive="both")        # 0–10 scale (we see 7, 8, 9, 10)
m_1to5  = s.between(1, 5,  inclusive="both")        # 1–5 stars (dominant in your sample)
m_pm10  = s.between(-10, 10, inclusive="both")      # -10…10 scale (captures 0 and negatives)

# Label the inferred source scale (useful for QA)
source = pd.Series("unknown", index=df.index, dtype="object")
source.loc[m_pct] = "0to100"
source.loc[m_0to10] = "0to10"
source.loc[m_1to5] = "1to5"
source.loc[m_pm10 & ~m_0to10 & ~m_1to5] = "pm10"
df["sentiment_source_scale"] = source

# Normalize each to [-1, 1]
u = pd.Series(np.nan, index=df.index, dtype="float")
u.loc[m_pct]                                   = s.loc[m_pct]   / 50.0 - 1.0    # 0→-1, 50→0, 100→1
u.loc[m_0to10]                                 = s.loc[m_0to10] / 5.0  - 1.0    # 0→-1, 5→0, 10→1
u.loc[m_1to5]                                  = (s.loc[m_1to5] - 3.0) / 2.0    # 1→-1, 3→0, 5→1
u.loc[m_pm10 & ~m_0to10 & ~m_1to5]             = s.loc[m_pm10 & ~m_0to10 & ~m_1to5] / 10.0
df["sentiment_norm"] = u.clip(-1, 1)

# Choose your target scale
TARGET = "1to5"     # options: "1to5" (default), "1to10", "pm5", "0to1"

def from_unit(u: pd.Series, target: str) -> pd.Series:
    if target == "pm5":     # -5 … 5, where 0 is neutral
        return 5.0 * u
    if target == "1to10":   # 1 … 10, where 5.5 is neutral
        return 1.0 + 9.0 * (u + 1.0) / 2.0
    if target == "0to1":    # 0 … 1
        return (u + 1.0) / 2.0
    if target == "1to5":    # 1 … 5, where 3 is neutral
        return 1.0 + 4.0 * (u + 1.0) / 2.0
    raise ValueError(f"Unknown target scale: {target}")

df[f"sentiment_{TARGET}"] = from_unit(df["sentiment_norm"], TARGET)

# Bin using the canonical [-1, 1] space (consistent regardless of target units)
bins   = [-1.01, -0.2, 0.2, 1.01]
labels = ["negative", "neutral", "positive"]
df["sentiment_bin"] = pd.cut(df["sentiment_norm"], bins=bins, labels=labels, include_lowest=True)


In [ ]:
df.value_counts("sentiment_norm")

sentiment_norm
 1.0    802
 0.5    695
-1.0    613
 0.0    466
-0.1    258
-0.5    130
 0.8     18
-0.2      5
 0.4      3
 0.6      3
 0.9      1
Name: count, dtype: int64

In [8]:
df.value_counts(["primary_category", "sentiment_bin"])

primary_category                   sentiment_bin
Sentiment/Review Meta              positive         526
Quality/Durability/Build           positive         399
Design/Features/Functionality      positive         337
Quality/Durability/Build           negative         242
Other/Uncategorized                neutral          196
Design/Features/Functionality      negative         194
                                   neutral          123
Size/Capacity/Organization/Space   negative         116
Shipping/Delivery/Fulfillment      neutral          103
Quality/Durability/Build           neutral           94
Size/Capacity/Organization/Space   positive          90
Sentiment/Review Meta              neutral           80
Price/Value/Cost                   positive          64
Sentiment/Review Meta              negative          59
Price/Value/Cost                   negative          40
                                   neutral           33
Size/Capacity/Organization/Space   neutral           31

In [9]:
sent_order = ["positive", "negative", "neutral"]

ct = (
    pd.crosstab(df["primary_category"], df["sentiment_bin"])
      .reindex(columns=sent_order, fill_value=0)
)
ct["Total"] = ct.sum(axis=1)
ct = ct.sort_values("Total", ascending=False)
ct.loc["Total"] = ct.sum()
ct


sentiment_bin,positive,negative,neutral,Total
primary_category,,,,
Quality/Durability/Build,399,242,94,735
Sentiment/Review Meta,526,59,80,665
Design/Features/Functionality,337,194,123,654
Size/Capacity/Organization/Space,90,116,31,237
Other/Uncategorized,15,23,196,234
Shipping/Delivery/Fulfillment,14,28,103,145
Price/Value/Cost,64,40,33,137
Customer Service/Warranty/Billing,28,13,14,55
Travel/Airline/Carry-on,22,7,10,39


In [18]:
reviews_path = supporting_files / "data" / "reviews"
file_path = reviews_path / 'binned_classifications_sentiment.json'

df.to_json(file_path, orient="records", indent=3)